In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import datetime
from src.get_rda_era5_2 import ERA5DataSource
import re

In [2]:
lat_dict = {
    "full": slice(50, 25),
    "small": slice(45, 30),
    "slgt_small": slice(50, 25),
    "slgt_full": slice(50, 25),
}
lon_dict = {
    "full": slice(360 - 125, 360 - 66),
    "small": slice(360 - 105, 360 - 85),
    "slgt_small": slice(360 - 125, 360 - 66),
    "slgt_full": slice(360 - 125, 360 - 66),
}
levels_dict = {
    "full": [925, 850, 700, 500, 300],
    "small": [925, 850, 700, 500, 300],
    "slgt_small": [925, 850, 700, 500, 300],
    "slgt_full": [925, 850, 700, 500, 300],
}
time_thin_dict = {"full": 1, "small": 6, "slgt_small": 6, "slgt_full": 1}
space_thin_dict = {"full": 1, "small": 4, "slgt_small": 4, "slgt_full": 1}

risk_level_dict = {
    "full": ["MDT", "HIGH"],
    "small": ["MDT", "HIGH"],
    "slgt_small": ["SLGT", "ENH", "MDT", "HIGH"],
    "slgt_full": ["SLGT", "ENH", "MDT", "HIGH"],
}

pressure_var_dict = {
    "full": ["geopotential", "specific_humidity", "temperature", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"],
    "small": ["geopotential", "specific_humidity", "temperature", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"],
    "slgt_small": ["geopotential", "specific_humidity", "temperature", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"],
    "slgt_full": ["geopotential", "specific_humidity", "temperature", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"],
}
surface_var_dict = {
    "full": ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"],
    "small": ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"],
    "slgt_small": ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"],
    "slgt_full": ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"],
}

In [3]:
detail = "slgt_full"

In [4]:
# --- risk days
pph = xr.load_dataset("data/raw_data/labelled_pph.nc")
missing_dates = [
    "200204250000", "200208300000", "200304150000", "200304160000",
    "200306250000", "200307270000", "200307280000", "200312280000",
    "200404140000", "200408090000", "200905280000", "201105210000",
    "202005240000", "200510240000",
]
dates_of_interest = pph["time"][pph["MAX_CAT"].isin(risk_level_dict[detail])]
dates_of_interest = dates_of_interest[dates_of_interest > "200203310000"]
dates_of_interest = dates_of_interest[~(dates_of_interest.isin(missing_dates))]
selected_days = pd.to_datetime(dates_of_interest.values, format="%Y%m%d%H%M").normalize()

In [5]:
LONG_TO_SHORT = {
    "10m_u_component_of_wind": "10u",
    "10m_v_component_of_wind": "10v",
    "2m_temperature": "2t",
    "2m_dewpoint_temperature": "2d",
    "geopotential": "z",
    "specific_humidity": "q",
    "temperature": "t",
    "u_component_of_wind": "u",
    "v_component_of_wind": "v",
    "vertical_velocity": "w",
}
SHORT_TO_LONG = {v: k for k, v in LONG_TO_SHORT.items()}

In [6]:
def longname_to_channel(longname, level=None):
    if longname in LONG_TO_SHORT:
        short = LONG_TO_SHORT[longname]
        return short if level is None else f"{short}{int(level)}"
    raise KeyError(f"No conversion known for {longname}; add to LONG_TO_SHORT")


def channels_for_config(cfg):
    return (
        [longname_to_channel(v) for v in surface_var_dict[cfg] if LONG_TO_SHORT.get(v, "")]
        + [longname_to_channel(v, lvl)
           for v in pressure_var_dict[cfg] if LONG_TO_SHORT.get(v, "")
           for lvl in levels_dict[cfg]]
    )


all_channels = channels_for_config(detail)

In [7]:
def channel_da_to_dataset(da):
    """
    Input da dims: (day, tod, channel, lat, lon)
    Output dataset:
      - surface vars: (day, tod, lat, lon)
      - pressure vars: (day, tod, level, lat, lon) or (level, day, tod, lat, lon) depending concat order
    """
    ds_out = {}

    for ch in da.channel.values:
        sub = da.sel(channel=ch).drop_vars("channel")

        if re.search(r"\d{3}$", ch):
            level = int(ch[-3:])          # IMPORTANT: int, not string
            longname = SHORT_TO_LONG[ch[:-3]]
            sub = sub.assign_coords(level=level).expand_dims("level")
            ds_out.setdefault(longname, []).append(sub)
        else:
            longname = SHORT_TO_LONG[ch]
            ds_out.setdefault(longname, []).append(sub)

    data_vars = {}
    for name, pieces in ds_out.items():
        if "level" in pieces[0].dims:
            merged = xr.concat(pieces, dim="level")
        else:
            # should be exactly one piece; if not, fail loudly
            if len(pieces) != 1:
                raise ValueError(f"Surface variable {name} produced {len(pieces)} pieces; expected 1.")
            merged = pieces[0]
        data_vars[name] = merged

    return xr.Dataset(data_vars)


def collect_one_day_fast(ds, day, detail):
    tods = list(range(0, 24, time_thin_dict[detail]))  # e.g. [0,6,12,18] or [0..23]
    return ds.get_12z_day_block(
        day=day.date(),                 # dt.date
        tods=tods,
        lat_slice=lat_dict[detail],
        lon_slice=lon_dict[detail],
        space_thin=space_thin_dict[detail],
        max_workers=8,
    )

In [8]:
ds = ERA5DataSource(all_channels)

In [9]:
for i, day in enumerate(selected_days):
    print(f"Processing {day.date()}")
    day_da = collect_one_day_fast(ds, day, detail)     # (day=1, tod, channel, lat, lon)
    day_ds = channel_da_to_dataset(day_da)
    break

Processing 2002-04-02


Process ForkProcess-7:
Process ForkProcess-1:
Process ForkProcess-3:
Process ForkProcess-8:
Process ForkProcess-4:
Process ForkProcess-6:
Process ForkProcess-5:
Process ForkProcess-2:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/glade/work/milesep/conda-envs/mlco/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/glade/work/milesep/conda-envs/mlco/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/glade/work/milesep/conda-envs/mlco/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/glade/work/milesep/conda-envs/mlco/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/glade/work/milesep/

KeyboardInterrupt: 

In [12]:
day_ds

<xarray.Dataset> Size: 78MB
Dimensions:                  (day: 1, lat: 101, lon: 237, tod: 24, level: 5)
Coordinates:
  * day                      (day) datetime64[ns] 8B 2002-04-02
  * lat                      (lat) float64 808B 50.0 49.75 49.5 ... 25.25 25.0
  * lon                      (lon) float64 2kB 235.0 235.2 235.5 ... 293.8 294.0
  * tod                      (tod) int64 192B 0 1 2 3 4 5 ... 18 19 20 21 22 23
  * level                    (level) int64 40B 300 500 700 850 925
Data variables:
    10m_u_component_of_wind  (day, tod, lat, lon) float32 2MB dask.array<chunksize=(1, 12, 101, 176), meta=np.ndarray>
    10m_v_component_of_wind  (day, tod, lat, lon) float32 2MB dask.array<chunksize=(1, 12, 101, 176), meta=np.ndarray>
    2m_dewpoint_temperature  (day, tod, lat, lon) float32 2MB dask.array<chunksize=(1, 12, 101, 176), meta=np.ndarray>
    2m_temperature           (day, tod, lat, lon) float32 2MB dask.array<chunksize=(1, 12, 101, 176), meta=np.ndarray>
    geopotential             (level, day, tod, lat, lon) float32 11MB dask.array<chunksize=(1, 1, 12, 101, 176), meta=np.ndarray>
    specific_humidity        (level, day, tod, lat, lon) float32 11MB dask.array<chunksize=(1, 1, 12, 101, 176), meta=np.ndarray>
    temperature              (level, day, tod, lat, lon) float32 11MB dask.array<chunksize=(1, 1, 12, 101, 176), meta=np.ndarray>
    u_component_of_wind      (level, day, tod, lat, lon) float32 11MB dask.array<chunksize=(1, 1, 12, 101, 176), meta=np.ndarray>
    v_component_of_wind      (level, day, tod, lat, lon) float32 11MB dask.array<chunksize=(1, 1, 12, 101, 176), meta=np.ndarray>
    vertical_velocity        (level, day, tod, lat, lon) float32 11MB dask.array<chunksize=(1, 1, 12, 101, 176), meta=np.ndarray>

In [11]:
day_ds.to_zarr('test.zarr')